# Notebook 3 — Non-CMF Unlearning on 3:7 Cross-Class Split

**Paper:** *An Illusion of Unlearning?* (Gao et al., AISTATS 2026 · arXiv 2604.08271)

**Evaluation — 3 metrics matching paper Table 1:**

| Metric | Description |
|--------|-------------|
| **Output** | Full model forward pass over sample index sets |
| **Linear Probe** | Freeze encoder → fresh linear head on full train features → test retain/forget acc |
| **NCC** | Freeze encoder → per-class mean features → nearest-class-center on test set |

**Methods:** NegGrad+, Random-label, SalUn, SCRUB, UNSIR, SVD  
**Prerequisite:** Notebook 1 output attached as Kaggle dataset. Set `CKPT_DATASET_DIR` below.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn')

In [ ]:
import os, sys, json, random, argparse, collections, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ══ SET THIS to the Kaggle dataset mount path from Notebook 1 ══
CKPT_DATASET_DIR = '/kaggle/input/cmf_benchmark-notebook1'  # ← EDIT

_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/cmf_benchmark/cmf_benchmark_config.json',
]
config_path = CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p; CKPT_ROOT_NB1 = os.path.dirname(_p); break
assert config_path, 'cmf_benchmark_config.json not found. Check CKPT_DATASET_DIR.'

with open(config_path) as f: CFG = json.load(f)

TEST_MODE         = CFG['TEST_MODE']
TEST_FRACTION     = CFG['TEST_FRACTION']
_MODE_TAG         = CFG['_MODE_TAG']
DATASET           = CFG['DATASET']
ARCH              = CFG['ARCH']
IS_VIT            = CFG['IS_VIT']
NUM_CLASSES       = CFG['NUM_CLASSES']
CLASS_LABEL_NAMES = CFG['CLASS_LABEL_NAMES']
SPLIT_SEEDS       = CFG['SPLIT_SEEDS']
FORGET_FRACTION   = CFG['FORGET_FRACTION']
PRETRAIN_LR       = CFG['PRETRAIN_LR']
PRETRAIN_EPOCHS   = CFG['PRETRAIN_EPOCHS']
PRETRAIN_BS       = CFG['PRETRAIN_BS']
PRETRAIN_PATIENCE = CFG['PRETRAIN_PATIENCE']
_TOTAL            = {DATASET: CFG['TOTAL']}
_PER_CLASS        = {DATASET: CFG['PER_CLASS']}

_old_root = CFG['CKPT_ROOT']
def _repath(p): return p.replace(_old_root, CKPT_ROOT_NB1)
CKPT_PRETRAIN = _repath(CFG['CKPT_PRETRAIN'])
SPLIT_DIR     = _repath(CFG['SPLIT_DIR'])

DATA_PATH     = '/kaggle/working/data'
CKPT_ROOT_NB3 = '/kaggle/working/checkpoints/cmf_benchmark_nb3'
os.makedirs(CKPT_ROOT_NB3, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

MAX_EPOCHS = 1 if TEST_MODE else 50
UNLEARN_BS = 8 if TEST_MODE else 128

print(f'Mode={"TEST" if TEST_MODE else "FULL"}  Dataset={DATASET}  Arch={ARCH}')
print(f'MAX_EPOCHS={MAX_EPOCHS}  Seeds={SPLIT_SEEDS}')
print(f'  [{ "OK" if os.path.exists(CKPT_PRETRAIN) else "MISSING"}] CKPT_PRETRAIN: {CKPT_PRETRAIN}')

In [ ]:
from utils import get_dataset, get_model, test, SubSet
from unlearn import unlear_func
import utils as _utils_module, functools

_orig_test = test
@functools.wraps(_orig_test)
def test(*a, verbose=False, **kw): return _orig_test(*a, verbose=verbose, **kw)
_utils_module.test = test

# ── LR / epoch tables (same as existing notebooks) ────────────────────
_LR = {
    'random_label':        {'cifar10': 1e-2, 'cifar100': 3e-3, 'tinyimagenet': 5e-4},
    'salun':               {'cifar10': 1e-2, 'cifar100': 3e-3, 'tinyimagenet': 5e-4},
    'grad_ascent_descent': {'cifar10': 1e-3, 'cifar100': 5e-5, 'tinyimagenet': 5e-4},
    'scrub':               {'cifar10': 1e-4, 'cifar100': 1e-3, 'tinyimagenet': 5e-3},
    'tarun':               {'cifar10': 2e-3, 'cifar100': 3e-5, 'tinyimagenet': 2e-5},
    'SVD':                 {'cifar10': 1e-2, 'cifar100': 1e-2, 'tinyimagenet': 1e-3},
}
_EPOCHS = {
    'random_label': 3, 'salun': 3, 'grad_ascent_descent': 3,
    'scrub': 3, 'tarun': 3, 'SVD': 50,
}
if TEST_MODE: _EPOCHS = {k: 1 for k in _EPOCHS}

_SVD = {
    'cifar10':      {'alpha_r': 100,  'alpha_f': 3,  'samples': 900,  'max_patches': 10000},
    'cifar100':     {'alpha_r': 1000, 'alpha_f': 30, 'samples': 990,  'max_patches': 10000},
    'tinyimagenet': {'alpha_r': 30,   'alpha_f': 10, 'samples': 999,  'max_patches': 10000},
}
if TEST_MODE: _SVD = {k: {'alpha_r':2,'alpha_f':1,'samples':4,'max_patches':10} for k in _SVD}

def get_lr(method): return _LR.get(method, {}).get(DATASET, 1e-3)
def get_epochs(method): return min(_EPOCHS.get(method, 1 if TEST_MODE else 3), MAX_EPOCHS)

def make_args(**ov):
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=42, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5, min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
        grad_norm_clip=1.0, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3, SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=False, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=False, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT,
        project_name='kaggle', group_name='cmf_benchmark_no_cmf',
    )
    d.update(ov)
    return argparse.Namespace(**d)

print('Helpers loaded.')

In [ ]:
args_base = make_args()
dataset_train, dataset_test = get_dataset(args_base)
print(f'Full dataset — Train={len(dataset_train)}  Test={len(dataset_test)}  Classes={NUM_CLASSES}')

if TEST_MODE:
    def _stratified_subset(ds, frac, seed=42):
        labels = (ds.targets if hasattr(ds, 'targets')
                  else [ds.dataset.targets[i] for i in ds.indices]
                  if hasattr(ds, 'indices') else [s[1] for s in ds.samples])
        rng = random.Random(seed)
        by_cls = collections.defaultdict(list)
        for i, l in enumerate(labels): by_cls[int(l)].append(i)
        kept = []
        for c in sorted(by_cls):
            pool = by_cls[c]; rng.shuffle(pool)
            kept.extend(pool[:max(1, math.ceil(len(pool)*frac))])
        sub = torch.utils.data.Subset(ds, kept)
        base_t = ds.targets if hasattr(ds, 'targets') else [ds.dataset.targets[i] for i in ds.indices]
        sub.targets = [base_t[i] for i in kept]
        return sub
    dataset_train = _stratified_subset(dataset_train, TEST_FRACTION)
    dataset_test  = _stratified_subset(dataset_test,  TEST_FRACTION)
    print(f'TEST_MODE: Train={len(dataset_train)}  Test={len(dataset_test)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

LOADER_KW = dict(batch_size=UNLEARN_BS, num_workers=2, pin_memory=True, shuffle=True, drop_last=True)
TEST_KW   = dict(batch_size=min(256, len(dataset_test)), num_workers=2, pin_memory=True, shuffle=False)
train_loader = torch.utils.data.DataLoader(
    dataset_train, batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True, shuffle=True, drop_last=True)
test_loader  = torch.utils.data.DataLoader(dataset_test, **TEST_KW)

splits = {}
for seed in SPLIT_SEEDS:
    sf = f'{SPLIT_DIR}/forget_indices_seed{seed}.json'
    assert os.path.exists(sf), f'Split file missing: {sf}'
    with open(sf) as f: splits[seed] = json.load(f)
    print(f'Seed {seed}: forget={splits[seed]["n_forget"]}  retain={splits[seed]["n_retain"]}')

# Load Θ_o
args_pt = make_args(unlearn_method='pre_train', remove_FC=False, CMFClassifier=False)
orig_model = get_model(args_pt, device)
orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))
orig_model.eval()
print('\n── Θ_o test accuracy ──')
test(orig_model, device, test_loader, [], CLASS_LABEL_NAMES, NUM_CLASSES, set_name='Test')

## Three-Metric Eval Harness (Index-Based)

Matches paper Section 3.1 (Output) and Section 3.2 (Linear Probe, NCC) exactly.
All three metrics use **sample index sets**, not class labels.

In [ ]:
# ── Metric 1: Output ─────────────────────────────────────────────────
def eval_output_on_indices(model, dataset, indices, device, batch_size=256):
    if len(indices) == 0: return float('nan')
    loader = torch.utils.data.DataLoader(
        SubSet(dataset, indices), batch_size=batch_size,
        shuffle=False, num_workers=2, pin_memory=True)
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += y.size(0)
    return correct / max(1, total)


# ── Feature extraction (shared by Probe and NCC) ─────────────────────
@torch.no_grad()
def _extract_features(model, loader, device):
    model.eval(); Xs, ys = [], []
    for x, y in loader:
        x = x.to(device)
        f = model.extract_features(x) if hasattr(model, 'extract_features') else model(x)
        Xs.append(f.cpu()); ys.append(y)
    return torch.cat(Xs, 0).float(), torch.cat(ys, 0).long()


def _test_split_30_70(yte_numpy):
    """Split test indices into forget (30%) / retain (70%) per class, seed=0."""
    rng = random.Random(0)
    by_cls = collections.defaultdict(list)
    for i, l in enumerate(yte_numpy): by_cls[int(l)].append(i)
    fgt, ret = [], []
    for c in sorted(by_cls):
        pool = list(by_cls[c]); rng.shuffle(pool)
        nf = max(1, round(len(pool) * 0.30))
        fgt.extend(pool[:nf]); ret.extend(pool[nf:])
    return ret, fgt


# ── Metric 2: Linear Probe ────────────────────────────────────────────
def eval_probe_on_indices(model, dataset_train, dataset_test,
                          retain_indices, forget_indices,
                          device, num_classes,
                          probe_epochs=100, probe_lr=0.01, batch_size=256):
    """
    Paper Section 3.2: train a fresh linear classifier on frozen encoder features
    extracted from the FULL training set (D_r ∪ D_f), then eval on test set.
    """
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)

    full_loader = torch.utils.data.DataLoader(
        dataset_train, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    Xtr, ytr = _extract_features(model, full_loader, device)

    te_loader = torch.utils.data.DataLoader(
        dataset_test, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    Xte, yte = _extract_features(model, te_loader, device)

    head = nn.Linear(Xtr.size(1), num_classes).to(device)
    opt  = optim.SGD(head.parameters(), lr=probe_lr, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for _ in range(probe_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()

    head.eval()
    with torch.no_grad(): pred = head(Xte.to(device)).argmax(1).cpu()
    ret_idx, fgt_idx = _test_split_30_70(yte.numpy())

    def _acc(idxs):
        if not idxs: return float('nan')
        return float((pred[idxs] == yte[idxs]).float().mean())

    for p in model.parameters(): p.requires_grad_(True)
    return _acc(ret_idx), _acc(fgt_idx)


# ── Metric 3: NCC ─────────────────────────────────────────────────────
def eval_ncc_on_indices(model, dataset_train, dataset_test,
                        retain_indices, forget_indices,
                        device, num_classes, batch_size=256):
    """
    Paper Section 2.1 eq.(5): classify by nearest class-mean in L2-normalized feature space.
    Class means computed from full training set.
    """
    model.eval()
    full_loader = torch.utils.data.DataLoader(
        dataset_train, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    Xtr, ytr = _extract_features(model, full_loader, device)
    Xtr_n = F.normalize(Xtr, dim=1)

    cmeans = torch.zeros(num_classes, Xtr.size(1))
    for c in range(num_classes):
        m = (ytr == c)
        if m.any(): cmeans[c] = Xtr_n[m].mean(0)
    cmeans_n = F.normalize(cmeans, dim=1)

    te_loader = torch.utils.data.DataLoader(
        dataset_test, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    Xte, yte = _extract_features(model, te_loader, device)
    Xte_n = F.normalize(Xte, dim=1)

    with torch.no_grad(): pred = (Xte_n @ cmeans_n.t()).argmax(1)
    ret_idx, fgt_idx = _test_split_30_70(yte.numpy())

    def _acc(idxs):
        if not idxs: return float('nan')
        return float((pred[idxs] == yte[idxs]).float().mean())

    return _acc(ret_idx), _acc(fgt_idx)


# ── Combined ──────────────────────────────────────────────────────────
def eval_three_metrics(model, dataset_train, dataset_test,
                       retain_indices, forget_indices,
                       device, num_classes,
                       run_probe=True, run_ncc=True, probe_epochs=100):
    out_r = eval_output_on_indices(model, dataset_train, retain_indices, device)
    out_f = eval_output_on_indices(model, dataset_train, forget_indices,  device)
    if run_probe and not TEST_MODE:
        pr_r, pr_f = eval_probe_on_indices(
            model, dataset_train, dataset_test, retain_indices, forget_indices,
            device, num_classes, probe_epochs=probe_epochs)
    else:
        pr_r = pr_f = float('nan')
    if run_ncc and not TEST_MODE:
        ncc_r, ncc_f = eval_ncc_on_indices(
            model, dataset_train, dataset_test, retain_indices, forget_indices,
            device, num_classes)
    else:
        ncc_r = ncc_f = float('nan')
    return dict(
        output_retain_acc=out_r, output_forget_acc=out_f,
        probe_retain_acc=pr_r,   probe_forget_acc=pr_f,
        ncc_retain_acc=ncc_r,    ncc_forget_acc=ncc_f,
    )

print('Three-metric eval harness defined (Output / Linear Probe / NCC).')

## C. Run Non-CMF Unlearning Methods

All methods: NegGrad+, Random-label, SalUn, SCRUB, UNSIR, SVD  
Cap: ≤ 50 epochs per method. Evaluated with all 3 paper metrics.

In [ ]:
RUN_METHODS = [
    'grad_ascent_descent',  # NegGrad+
    'random_label',
    'salun',
    'scrub',
    'tarun',                # UNSIR
    'SVD',
]
NAME_MAP = {
    'grad_ascent_descent': 'NegGrad+',
    'random_label': 'Random-label',
    'salun': 'SalUn',
    'scrub': 'SCRUB',
    'tarun': 'UNSIR',
    'SVD': 'SVD',
}
print(f'Methods: {RUN_METHODS}')
print(f'Seeds: {SPLIT_SEEDS}')
print(f'Total runs: {len(RUN_METHODS)} × {len(SPLIT_SEEDS)} = {len(RUN_METHODS)*len(SPLIT_SEEDS)}')

In [ ]:
def run_unlearn_cmf_benchmark(method, seed, split):
    retain_indices = split['retain_indices']
    forget_indices = split['forget_indices']
    lr     = get_lr(method)
    epochs = get_epochs(method)

    kw = dict(
        unlearn_method=method,
        epochs_or_steps=epochs, lr=lr,
        batch_size=UNLEARN_BS,
        num_retain_samples=len(retain_indices),
        num_forget_samples=len(forget_indices),
        unlearn_class=[],  # cross-class split: no whole class removed
        remove_FC=False, CMFClassifier=True,
        grad_norm_clip=1.0,
    )
    if method == 'salun':  kw['salun_threshold'] = 0.5
    if method == 'SVD':
        s = _SVD[DATASET]
        kw.update(SVD_alpha_r=s['alpha_r'], SVD_alpha_f=s['alpha_f'],
                  SVD_samples=s['samples'], SVD_max_patches=s['max_patches'])
    if method == 'tarun':
        kw.update(tarun_impair_lr=1e-4 if DATASET=='cifar10' else 2e-4,
                  tarun_samples_per_class=1000)
    if method == 'scrub':
        kw.update(scrub_del_bsz=64, scrub_sgda_bsz=64,
                  scrub_msteps=2, scrub_epochs=epochs)

    args = make_args(**kw)

    retain_ds = SubSet(dataset_train, retain_indices)
    forget_ds = SubSet(dataset_train, forget_indices)
    retain_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)
    forget_loader = torch.utils.data.DataLoader(forget_ds, **LOADER_KW)

    m = get_model(args, device)
    m.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))
    optimizer = optim.SGD(m.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)

    print(f'\n{"-"*60}')
    print(f'  {NAME_MAP.get(method, method)}  seed={seed}  lr={lr}  epochs={epochs}')
    print(f'  retain={len(retain_indices)}  forget={len(forget_indices)}')
    print(f'{"-"*60}')

    t0 = time.time()
    try:
        unlearnt = unlear_func[method](
            args=args, model=m, device=device,
            retain_loader=retain_loader,
            forget_loader=forget_loader,
            train_loader=train_loader,
            val_loader=None,
            test_loader=test_loader,
            optimizer=optimizer,
            epochs=epochs,
            train_dataset=dataset_train,
            val_index=np.arange(len(dataset_train)),
            test_forget_loader=forget_loader,
        )
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'  ERROR: {e}')
        return None

    elapsed = time.time() - t0

    ckpt_dir = f'{CKPT_ROOT_NB3}/{method}'
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_out = f'{ckpt_dir}/{DATASET}_{ARCH}_{_MODE_TAG}_seed{seed}.pt'
    torch.save(unlearnt.state_dict(), ckpt_out)

    metrics = eval_three_metrics(
        unlearnt, dataset_train, dataset_test,
        retain_indices, forget_indices, device, NUM_CLASSES,
        run_probe=not TEST_MODE, run_ncc=not TEST_MODE, probe_epochs=100)

    print(f'  → Output  R={metrics["output_retain_acc"]:.4f}  F={metrics["output_forget_acc"]:.4f}')
    print(f'  → Probe   R={metrics["probe_retain_acc"]:.4f}  F={metrics["probe_forget_acc"]:.4f}')
    print(f'  → NCC     R={metrics["ncc_retain_acc"]:.4f}  F={metrics["ncc_forget_acc"]:.4f}')
    print(f'  → wall clock: {elapsed/60:.1f} min')

    return dict(method=method, method_name=NAME_MAP.get(method, method), seed=seed,
                **metrics, wall_clock_minutes=elapsed/60, ckpt=ckpt_out)


print('run_unlearn_cmf_benchmark defined.')

In [ ]:
ALL_RESULTS = []

for seed in SPLIT_SEEDS:
    split = splits[seed]
    print(f'\n{"#"*65}')
    print(f'  SEED {seed}  — forget={split["n_forget"]}  retain={split["n_retain"]}')
    print(f'{"#"*65}')
    for method in RUN_METHODS:
        res = run_unlearn_cmf_benchmark(method, seed, split)
        if res: ALL_RESULTS.append(res)

print(f'\nAll methods done. {len(ALL_RESULTS)} runs completed.')

## D. Results — CSV + Summary Table (paper Table 1 format)

In [ ]:
results_df = pd.DataFrame(ALL_RESULTS)

METRIC_COLS = ['output_retain_acc', 'output_forget_acc',
               'probe_retain_acc',  'probe_forget_acc',
               'ncc_retain_acc',    'ncc_forget_acc']
present_cols = [c for c in METRIC_COLS if c in results_df.columns]

# Mean ± std over seeds per method
agg = results_df.groupby('method_name')[present_cols].agg(['mean','std'])

print(f'\n=== Non-CMF Results — {DATASET}/{ARCH}')
print(f'    mean ± std over {len(SPLIT_SEEDS)} seeds, matches paper Table 1 format ===')
print(f'{"Method":20s}  {"Out-R":>8} {"Out-F":>8}  {"Prb-R":>8} {"Prb-F":>8}  {"NCC-R":>8} {"NCC-F":>8}')
print('-'*80)
for name in [NAME_MAP[m] for m in RUN_METHODS if NAME_MAP[m] in agg.index]:
    row = agg.loc[name]
    def _fmt(col):
        try:
            m = row[(col,'mean')]; s = row[(col,'std')]
            return f'{m:.3f}±{s:.3f}' if not (pd.isna(m) or pd.isna(s)) else '  N/A   '
        except: return '  N/A   '
    print(f'{name:20s}  {_fmt("output_retain_acc")}  {_fmt("output_forget_acc")}  '
          f'{_fmt("probe_retain_acc")}  {_fmt("probe_forget_acc")}  '
          f'{_fmt("ncc_retain_acc")}  {_fmt("ncc_forget_acc")}')

csv_path = f'/kaggle/working/results_no_cmf_{DATASET}_{ARCH}.csv'
results_df.to_csv(csv_path, index=False)
print(f'\nFull results saved: {csv_path}')

# ── Bar chart: output retain vs forget per method ─────────────────────
df_plot = results_df.groupby('method_name')[['output_retain_acc','output_forget_acc']].mean()
ordered = [NAME_MAP[m] for m in RUN_METHODS if NAME_MAP[m] in df_plot.index]
df_plot = df_plot.reindex(ordered)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(df_plot)); w = 0.38
ax.bar(x - w/2, df_plot['output_retain_acc'], w, label='Retain', color='steelblue', alpha=0.85)
ax.bar(x + w/2, df_plot['output_forget_acc'], w, label='Forget', color='tomato',    alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(ordered, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Output Accuracy'); ax.set_ylim(0, 1.08)
ax.set_title(f'Non-CMF Methods — Output Accuracy\n'
             f'{DATASET}/{ARCH} (mean over {len(SPLIT_SEEDS)} seeds)')
ax.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/chart_no_cmf.png', dpi=120)
plt.show()
print('Chart saved.')